# 3. Gains, gauge and total variation

Calibration solves the measurement equation $V_{pq} = G_p M_{pq} G_q^H$ for the
2×2 complex gain $G_p$ of every antenna in every **solution interval** (a bin of
integrations × channels). Rang's core runs a port of QuartiCal's `complex`
(full-Jones) solver, bit-for-bit, then computes residuals and held-out losses in
one pass over the data.

Three ideas this notebook makes visible:

1. gains are only defined up to a **unitary gauge** (and what that means for
   plots);
2. the **total variation (TV)** of the gains across time and frequency, which the
   project's flagging experiment uses as a smoothness signal;
3. the **held-out error**: how well the gains predict cells they were not fitted
   to, which is how every choice here is judged.

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores; set before the first core call

import numpy as np
import pandas as pd

from rangtoy.notebook import data, experiment, flags, gains, image, steer, survey

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Downloads and the extracted fixture go here; choose your own with RANG_NOTEBOOK_CACHE.
CACHE = data.default_cache_dir()
# LUNO sections run when a local copy of the slice exists (RANG_NOTEBOOK_LUNO=0 turns them off).
LUNO = os.environ.get("RANG_NOTEBOOK_LUNO", "1") != "0" and data.find_luno_slice() is not None
print("cache:", CACHE, "| LUNO slice available:", LUNO)

In [ ]:
ds = data.load_fixture(cache_dir=CACHE)
pd.Series(data.summary(ds), name="value").to_frame()

In [ ]:
sol = gains.solve(ds, time_interval=2, freq_interval=32, iterations=50)
gains.losses_table(sol).T

**Reading the table.** MSE is the weighted squared residual per cell, $\sum w|V-GMG^H|^2$ over
the cells of each split divided by their number. With correct weights and a
perfect model it would be about 8 per cell (four complex correlations, with
`WEIGHT_SPECTRUM` as inverse variance per real part); it is larger here because
the point-source model ignores the other sources in the field and the weights
are not exact. Validation and test are close to train: the gains are not
over-fitted. "Invalid" counts held-out cells whose gain bin has no valid
solution; such cells make the loss unavailable instead of being silently dropped.

## Amplitude and phase maps

One panel per antenna, solution time bins down, frequency bins across. Grey bins
have no valid solution (no data, or a singular reference).

In [ ]:
gains.gain_maps(ds, sol, "amplitude", "XX")

In [ ]:
gains.gain_maps(ds, sol, "phase", "XX")

## The unitary gauge caveat

Full-Jones gains are not unique. The cell below prints the explanation and shows
it on one antenna.

In [ ]:
print(gains.GAUGE_NOTE)
gains.gauge_figure(ds, sol, antenna=1)

**What to look for:** the left panel's phase changes when only the gauge
changes; the right panel's Frobenius norm $\|G\|_F$ is identical in both. Never
interpret a raw full-Jones phase without saying which gauge it is in.

## Total variation

The TV of the gains sums the size of the jumps $\|G_{e^+} - G_{e^-}\|_F$ between
neighbouring solution bins, in time and in frequency, scaled by the median gain
norm and by the bin spacing. A TV **map** gives each bin the mean of its
incident jumps. High TV can mean RFI-corrupted solutions, but also genuine
bandpass structure, low signal-to-noise, model error or solver failure: TV
alone is not an RFI label.

In [ ]:
gains.gain_maps(ds, sol, "tv")

In [ ]:
gains.gain_spectra(ds, sol, time_bin=0)

The residuals after the solve are the other half of the evidence: the
experiment's policy only flags a cell when its whitened residual is an outlier,
and TV only lowers the threshold locally.

In [ ]:
gains.residual_waterfall(ds, sol, ds.baselines()[0])

## Solution intervals

Longer intervals average more data per solution (less noise) but cannot follow
fast gain changes; shorter ones follow the gains but fit noise. The held-out
error arbitrates. Note: the split reserved two training cells per baseline in
every 2 × 32 bin; with other intervals some held-out cells can lose their
gain, which shows up as invalid predictions.

In [ ]:
trials = {f"{t} int x {f} ch": gains.solve(ds, None, t, f, 50)
          for t, f in [(1, 16), (2, 32), (4, 64), (7, 256)]}
gains.losses_table(trials)

## The iteration cap is not convergence

The frozen campaigns stop at 50 iterations. The simplification review found that
held-out error settles within a few iterations while the gains themselves keep
moving (research/simplification-review.md, section 2.7). The sweep shows held-out
MSE, the converged fraction and the largest change of any gain relative to the
last cap.

In [ ]:
sweep = gains.iteration_sweep(ds, caps=(1, 2, 4, 8, 12, 20, 50, 80))
display(sweep)
gains.plot_sweep(sweep)

### What a professional checks before trusting gains

- the converged fraction (here and in the LUNO four-station run about 98 %;
  only 83 % on the whole array);
- invalid predictions on validation and test;
- that features in the maps are in $\|G\|_F$ or gauge-fixed products, not raw phases;
- that a change of interval or cap that lowers TV does not raise held-out error.

## Steer it yourself

In [ ]:
panel = steer.SolvePanel(ds)
panel

In [ ]:
result = panel.update(time_interval=4, freq_interval=64, iterations=20, quantity="tv")
display(result["losses"])
result["maps"]